# Notebook Replica Set (INF-239)

Registro de las ejecuciones realizadas para la Tarea 3 de INF-239. Cada seccion corresponde a un requisito y muestra la evidencia capturada.

### Preparacion rapida del notebook

- Se importan las librerias basicas.
- Se definen las rutas al dataset y al JSON exportado.
- Se declara `run_cmd` para repetir los comandos tal como se corrieron en la terminal.

In [1]:
from pathlib import Path
from datetime import datetime
import json
import subprocess

PROJECT_ROOT = Path('.').resolve()
DATASET_DIR = PROJECT_ROOT / 'DiscursosOriginales'
EXPORT_PATH = PROJECT_ROOT / 'discursos-final-202373011-4-202304074-6.json'
CONNECTION_STRING = 'mongodb://mongo1:30001,mongo2:30002,mongo3:30003/?replicaSet=my-replica-set&readPreference=primary&appname=MongoDB%20Compass&ssl=false'

print(f'Proyecto: {PROJECT_ROOT}')
print(f'Dataset: {DATASET_DIR} | Archivos: {len(list(DATASET_DIR.glob("*.txt")))}')
print(f'Ultima actualizacion del JSON final: {datetime.fromtimestamp(EXPORT_PATH.stat().st_mtime)}')

def run_cmd(cmd, check=False):
    print('$', ' '.join(cmd))
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.stdout.strip():
        print(proc.stdout.strip())
    if proc.stderr.strip():
        print(proc.stderr.strip())
    if check and proc.returncode != 0:
        raise RuntimeError(proc.stderr)
    return proc


Proyecto: C:\Users\sofia\OneDrive\Escritorio\BDD\Tarea 3\Tarea 3\tarea3-bd
Dataset: C:\Users\sofia\OneDrive\Escritorio\BDD\Tarea 3\Tarea 3\tarea3-bd\DiscursosOriginales | Archivos: 680
Ultima actualizacion del JSON final: 2025-11-06 19:26:37.520532


### Requisito 1: Replica Set en Docker (3 nodos)

Se despliega la arquitectura con `docker compose -f deploy-202373011-4-202304074-6.yml up -d`, dejando registro de la ejecucion y de cualquier mensaje que entrega Docker.

In [2]:
# 1) Despliegue de los contenedores (requiere Docker Desktop corriendo)
result = run_cmd(['docker', 'compose', '-f', 'deploy-202373011-4-202304074-6.yml', 'up', '-d'])
if result.returncode != 0:
    print('[WARN] Inicia Docker Desktop y vuelve a ejecutar la celda.')


$ docker compose -f deploy-202373011-4-202304074-6.yml up -d
[+] Running 4/4
 ✓ Container mongo1  Started
 ✓ Container mongo2  Started
 ✓ Container mongo3  Started
 ✓ Container rs-init  Started


In [3]:
# 2) Verificar el estado del Replica Set
from pymongo import MongoClient
try:
    client = MongoClient(CONNECTION_STRING, serverSelectionTimeoutMS=2000)
    status = client.admin.command('hello')
    print(f"Replica Set: {status.get('setName')} | Primario: {status.get('primary')}")
    print('Miembros:', [m['name'] for m in status.get('hosts', [])])
except Exception as exc:
    print('No fue posible consultar el Replica Set. Mensaje recibido:')
    print(exc)
    print('Ejecuta la celda nuevamente cuando los contenedores esten activos.')


Replica Set: my-replica-set | Primario: mongo1:30001
Miembros: ['mongo1:30001', 'mongo2:30002', 'mongo3:30003']


### Requisito 2: Preprocesamiento e ingesta del corpus

- **Script usado:** `conectarMongo.py`.
- `python conectarMongo.py --ingest --skip-db` genera el JSON final (`discursos-final-202373011-4-202304074-6.json`) y confirma que existen 680 discursos.
- `python conectarMongo.py --ingest` (sin `--skip-db`) inserta los documentos en `Politica.Discursos` utilizando el Replica Set.
- El notebook muestra los mensajes completos de ambas ejecuciones para respaldar el proceso.

In [4]:
# 3) Ingesta del corpus (quitar --skip-db cuando Mongo este arriba)
ingest = run_cmd(['python', 'conectarMongo.py', '--ingest', '--skip-db'])


$ python conectarMongo.py --ingest --skip-db


Archivo JSON generado en discursos-final-202373011-4-202304074-6.json
Batches: 100%|##########| 43/43 [00:12<00:00,  3.39it/s]


In [5]:
# 4) Inspeccion del JSON final generado
with EXPORT_PATH.open('r', encoding='utf-8') as handler:
    docs = json.load(handler)

print(f'Se procesaron {len(docs)} discursos.')
print('Ejemplo de documento:')
example = docs[0]
print({'_id': example['_id'], 'archivo': example['metadata']['filename']})
print('Primeros 180 caracteres del texto original:')
print(example['texto'][:180].replace('\n', ' ') + '...')


Se procesaron 680 discursos.
Ejemplo de documento:
{'_id': '44b5a3c1bb8f59b2593341223422c48ed4f78fb31ff70a0781751434b44419cd', 'archivo': '100020.txt'}
Primeros 180 caracteres del texto original:
Muy buenos días:     Disfruten de este Sol de invierno. Gabriela Mistral decía que “hay que amar a nuestras ciudades” y agregaba que la belleza de nuestras ciudades nos ennoblece y...


### Requisito 3: Consulta semantica que devuelve top 5

El mismo script `conectarMongo.py` implementa la consulta pedida. Con el JSON cargado se ejecuta `conectarMongo.py --query "Educacion publica y crecimiento economico"`. El notebook ilustra como el modelo `all-MiniLM-L6-v2` calcula el embedding de la consulta, obtiene la similitud coseno y lista los cinco discursos con mayor puntaje.

In [6]:
# 5) Consulta semantica offline (usa el JSON exportado para mostrar resultados)
import numpy as np
from sentence_transformers import SentenceTransformer

query = 'Educacion publica y crecimiento economico'
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
query_emb = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0]
emb_matrix = np.array([doc['embedding'] for doc in docs], dtype=np.float32)
scores = emb_matrix @ query_emb
ordered = np.argsort(-scores)[:5]
for position, idx in enumerate(ordered, start=1):
    doc = docs[idx]
    snippet = doc['texto'][:160].replace('\n', ' ')
    print(f"[{position}] {doc['metadata']['filename']} | score={scores[idx]:.4f}")
    print(snippet + '...')
    print('-' * 70)


C:\Users\sofia\AppData\Local\Programs\Python\Python313\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


[1] 82157.txt | score=0.6314
Muy buenos días:     Quiero saludar con mucho cariño a la ministra de Educación, al presidente de la FIDE, a los directivos, pero principalmente a todos ustedes...
----------------------------------------------------------------------
[2] 80522.txt | score=0.5985
Muy buenos días:     La verdad es que aquí se ve el espíritu de innovación, de emprendimiento, cómo ha cambiado este barrio, cómo está mejorando la calidad de n...
----------------------------------------------------------------------
[3] 97967.txt | score=0.5660
Muchas gracias Primer Ministro Abe, por esta cálida bienvenida.     Tanto la teoría económica como la historia económica nos demuestran en forma elocuente que h...
----------------------------------------------------------------------
[4] 86296.txt | score=0.5638
Muy buenos días:     Hoy hemos recibido, de parte de este grupo de trabajo, presidido por la ex ministra Mariana Aylwin, este Informe “Todos al Aula”, que es un...
---------------

### Requisito 4: Robustez del sistema

- **Consistencia**: se ejecuta `countDocuments()` en cada nodo para comprobar que el numero de registros coincide.
- **Alta disponibilidad**: se detiene `mongo1` mediante `docker stop` y se evidencia el cambio de rol antes de volver a iniciarlo.
- Toda la salida se deja tal cual aparece para respaldar las pruebas.

In [8]:
# 6) Consistencia: conteo de documentos replicados en los tres nodos
commands = [
    ["docker", "exec", "mongo1", "mongosh", "--port", "30001", "--quiet", "--eval", "db.getSiblingDB('Politica').Discursos.countDocuments()"],
    ["docker", "exec", "mongo2", "mongosh", "--port", "30002", "--quiet", "--eval", "db.getSiblingDB('Politica').Discursos.countDocuments()"],
    ["docker", "exec", "mongo3", "mongosh", "--port", "30003", "--quiet", "--eval", "db.getSiblingDB('Politica').Discursos.countDocuments()"],
]
for cmd in commands:
    run_cmd(cmd, check=True)


$ docker exec mongo1 mongosh --port 30001 --quiet --eval db.getSiblingDB('Politica').Discursos.countDocuments()
679
$ docker exec mongo2 mongosh --port 30002 --quiet --eval db.getSiblingDB('Politica').Discursos.countDocuments()
679
$ docker exec mongo3 mongosh --port 30003 --quiet --eval db.getSiblingDB('Politica').Discursos.countDocuments()
679


In [9]:
# 7) Alta disponibilidad: detener el primario y recuperar el cluster
import time

run_cmd(["docker", "stop", "mongo1"], check=True)
run_cmd(["docker", "exec", "mongo2", "mongosh", "--port", "30002", "--quiet", "--eval", "rs.status().members.map(m => ({name: m.name, state: m.stateStr}))"], check=True)
run_cmd(["python", "conectarMongo.py", "--query", "Mensaje presidencial sobre educacion publica"], check=True)
run_cmd(["docker", "start", "mongo1"], check=True)
time.sleep(8)
run_cmd(["docker", "exec", "mongo2", "mongosh", "--port", "30002", "--quiet", "--eval", "rs.status().members.map(m => ({name: m.name, state: m.stateStr}))"], check=True)


$ docker stop mongo1
mongo1
$ docker exec mongo2 mongosh --port 30002 --quiet --eval rs.status().members.map(m => ({name: m.name, state: m.stateStr}))
[
  { name: 'mongo1:30001', state: '(not reachable/healthy)' },
  { name: 'mongo2:30002', state: 'PRIMARY' },
  { name: 'mongo3:30003', state: 'SECONDARY' }
]
$ python conectarMongo.py --query Mensaje presidencial sobre educacion publica
Top 5 resultados para la consulta: "Mensaje presidencial sobre educacion publica"

[1] ID: a6458b4fa6d85ad7d370dc28cf0f9425d5b4f4724447d39ab95a781bd5145801 | archivo: 82157.txt
   Puntaje coseno: 0.5579
   Fragmento: Muy buenos días:     Quiero saludar con mucho cariño a la ministra de Educación, al presidente de la FIDE, a los directivos, pero principalmente a todos ustedes, que son los que durante muchas décadas le han dado vida, a...
--------------------------------------------------------------------------------
[2] ID: 14fb5c3abf3adc8011c1dd76f557a310f18f64457d12880d072b3eee3d9ef093 | archivo: 77982

In [7]:
# 6) Limpieza opcional de la infraestructura
cleanup = run_cmd(['docker', 'compose', '-f', 'deploy-202373011-4-202304074-6.yml', 'down'])
if cleanup.returncode != 0:
    print('Si los contenedores no estaban arriba, este mensaje se puede ignorar.')


$ docker compose -f deploy-202373011-4-202304074-6.yml down


time="2025-11-06T19:34:23-03:00" level=warning msg="C:\\Users\\sofia\\OneDrive\\Escritorio\\BDD\\Tarea 3\\Tarea 3\\tarea3-bd\\deploy-202373011-4-202304074-6.yml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"
error during connect: Get "http://%2F%2F.%2Fpipe%2FdockerDesktopLinuxEngine/v1.51/containers/json?all=1&filters=%7B%22label%22%3A%7B%22com.docker.compose.config-hash%22%3Atrue%2C%22com.docker.compose.oneoff%3DFalse%22%3Atrue%2C%22com.docker.compose.project%3Dtarea3-bd%22%3Atrue%7D%7D": open //./pipe/dockerDesktopLinuxEngine: The system cannot find the file specified.
Si los contenedores no estaban arriba, este mensaje se puede ignorar.
